# Вейвлет-анализ вариабельности сердечного ритмаАнализ ВСР через непрерывное вейвлет-преобразование (CWT) с базисным вейвлетом Морле.В отличие от FFT, вейвлет-преобразование даёт **частотно-временное** представление —позволяет отслеживать, как мощность в HF / LF / VLF меняется во времени, а не толькодавать одну усреднённую оценку на всю запись.## Что вычисляем| Раздел | Метод ||---|---|| 1 | Подготовка сигнала: интерполяция RR на равномерную сетку (4 Гц) || 2 | Прямое CWT (Морле): матрица W(a, b) — частотно-временной спектр || 3 | Скалограмма \|W(a, b)\|² с log-шкалой частоты || 4 | Динамика мощности в диапазонах: HF(t), LF(t), VLF(t) || 5 | Спектральные показатели: HF, LF, VLF, TP; HFn, LFn, VLFn (%); LF/HF || 6 | Стандартные отклонения: SDHF, SDLF, SDVLF, SDTP || 7 | Сравнение FFT vs Wavelet |## Зачем это нужно для wearableВСР с браслета — **нестационарный** сигнал. Усреднённый FFT теряет события,происходящие на коротких интервалах (стресс, эктопические сокращения, переходныепроцессы). CWT сохраняет временную локализацию — это критично для непрерывногомониторинга функционального состояния.

## 0. Импорты и загрузка

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport pywtfrom scipy.interpolate import CubicSplinefrom matplotlib.colors import LogNormfrom pathlib import Pathplt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})DATA_PATH = Path("data/rr_intervals.csv")FIGURES_DIR = Path("figures")FIGURES_DIR.mkdir(exist_ok=True)df = pd.read_csv(DATA_PATH, header=None, names=["RR"], comment="#")rr = df["RR"].to_numpy(dtype=float)n = len(rr)M = rr.mean()print(f"Загружено: {n} ударов, средний RR = {M:.1f} мс, HR = {n*60_000/rr.sum():.1f} уд/мин")

---## 1. Подготовка сигнала: интерполяция на равномерную сеткуRR-интервалы — это последовательность с **нерегулярным шагом по времени**: междуударами проходит разное число миллисекунд. Для CWT нужен равномерный временной ряд.Используем кубическую сплайн-интерполяцию с шагом dt = 250 мс (fs = 4 Гц).Это даёт верхнюю границу анализируемых частот fs/2 = 2 Гц — с большим запасом для HF (до 0.4 Гц).

In [ ]:
fs = 4.0          # Гцdt = 1.0 / fs     # 250 мсt_irr  = np.cumsum(rr) / 1000.0      # нерегулярная ось, сt_uni  = np.arange(t_irr[0], t_irr[-1], dt)cs     = CubicSpline(t_irr, rr)rr_uni = cs(t_uni)N = len(rr_uni)print(f"Точек на равномерной сетке: {N}")print(f"Длительность: {t_uni[-1]:.1f} с ({t_uni[-1]/60:.1f} мин)")print(f"Шаг dt: {dt*1000:.0f} мс, fs = {fs} Гц")

---## 2. Прямое вейвлет-преобразование (CWT)Базисная функция: **комплексный вейвлет Морле**.В PyWavelets его обозначение — `'morl'` (для CWT через `pywt.cwt`).Связь масштаба `a` с частотой `f`:$$f = \frac{f_c}{a \cdot dt}$$где `f_c` — центральная частота вейвлета.Шкала масштабов выбирается **логарифмически** между граничными частотами анализа.Это даёт равномерное разрешение по log(f), что физиологически корректно:HF, LF, VLF — это полосы примерно одинаковой ширины в логарифмической шкале.

In [ ]:
def get_wavelets(signal, w, f1, f2, D, n_scales=100):    '''    Прямое и обратное CWT в диапазоне частот [f1, f2].    Параметры    ---------    signal   : 1D-сигнал (равномерные отсчёты)    w        : имя вейвлета (например, 'morl', 'gaus7', 'db8')    f1, f2   : нижняя и верхняя частоты, Гц    D        : шаг дискретизации, с    n_scales : число масштабов в логарифмической сетке    Возвращает    ----------    cwt   : коэффициенты прямого преобразования, (n_scales, n_samples)    icwt  : взвешенные коэффициенты обратного CWT (для частотных полос)    F     : массив частот, соответствующих каждому масштабу    '''    fc = pywt.central_frequency(w)    a1 = fc / (D * f2)   # минимальный масштаб → высокая частота    a2 = fc / (D * f1)   # максимальный масштаб → низкая частота    A = np.exp(np.linspace(np.log(a1), np.log(a2), n_scales))    F = fc / (D * A)    dA = np.diff(A).tolist()    dA.append(dA[-1])    dA = np.array(dA)    cwt, _ = pywt.cwt(signal - np.mean(signal), A, w, sampling_period=D)    icwt = (cwt.T * (dA / (A ** 1.5))).T    return cwt, icwt, Fcwt, icwt, F = get_wavelets(rr_uni, 'morl', 0.015, 0.4, dt)print(f"CWT: {cwt.shape}  (n_scales × n_samples)")print(f"Частоты: от {F.min():.4f} до {F.max():.4f} Гц")

---## 3. Скалограмма \|W(a, b)\|²Матрица `|W|²` — это **мгновенное распределение энергии сигнала по частотам**.Каждый столбец — спектр в момент времени `b`, каждая строка — динамика на фиксированной частоте.Логарифмическая шкала по частоте + log-цвет интенсивности — стандарт для физиологии.

In [ ]:
power = np.abs(cwt) ** 2bands = {'HF': (0.15, 0.40), 'LF': (0.04, 0.15), 'VLF': (0.015, 0.04)}fig, ax = plt.subplots(figsize=(13, 6))order = np.argsort(F)extent = [t_uni[0], t_uni[-1], F[order[0]], F[order[-1]]]im = ax.imshow(power[order], aspect='auto', extent=extent,               cmap='jet', origin='lower',               norm=LogNorm(vmin=max(power.min(), 1e-3), vmax=power.max()))ax.set_yscale('log')ax.set_ylabel('Частота, Гц')ax.set_xlabel('Время, с')ax.set_title('Скалограмма — вейвлет-спектр |W(a,b)|² (Морле)', fontsize=14, fontweight='bold')for lab, (flo, fhi) in bands.items():    ax.axhline(flo, color='white', linewidth=0.8, linestyle='--', alpha=0.7)    ax.text(t_uni[-1] * 1.005, np.sqrt(flo * fhi), lab,            color='white', fontsize=11, fontweight='bold', va='center')ax.axhline(bands['HF'][1], color='white', linewidth=0.8, linestyle='--', alpha=0.7)plt.colorbar(im, ax=ax, label='Мощность, мс² (log)')plt.tight_layout()fig.savefig(FIGURES_DIR / 'fig1_scaleogram.png')plt.show()

---## 4. Динамика мощности по диапазонам HF(t), LF(t), VLF(t)Главное преимущество CWT — каждая частотная полоса даёт **временной ряд** мощности,а не одно число, как в FFT.Для оценки `HF(t)` суммируем (интегрируем) `|W(a, b)|²` по масштабам, попадающим в HF.То же для LF и VLF.

In [ ]:
def band_timeseries(power, F, flo, fhi):    mask = (F >= flo) & (F <= fhi)    f_sel = F[mask]    p_sel = power[mask]    order = np.argsort(f_sel)    return np.trapezoid(p_sel[order], f_sel[order], axis=0)HF_t  = band_timeseries(power, F, *bands['HF'])LF_t  = band_timeseries(power, F, *bands['LF'])VLF_t = band_timeseries(power, F, *bands['VLF'])TP_t  = HF_t + LF_t + VLF_tfig, ax = plt.subplots(figsize=(13, 5))ax.plot(t_uni, HF_t,  color='tomato',    linewidth=1.5, label=f'HF(t)')ax.plot(t_uni, LF_t,  color='limegreen', linewidth=1.5, label=f'LF(t)')ax.plot(t_uni, VLF_t, color='orange',    linewidth=1.5, label=f'VLF(t)')ax.set_xlabel('Время, с'); ax.set_ylabel('Мощность, мс²')ax.set_title('Динамика мощности в диапазонах HF / LF / VLF', fontsize=14, fontweight='bold')ax.legend(fontsize=11); ax.grid(alpha=0.3)plt.tight_layout()fig.savefig(FIGURES_DIR / 'fig2_band_timeseries.png')plt.show()

---## 5. Интегральные показатели| Показатель | Формула ||---|---|| HF, LF, VLF | среднее значение HF(t), LF(t), VLF(t) || TP (Total Power) | HF + LF + VLF || HFn, LFn, VLFn (%) | доля от TP || LF/HF | индекс вегетативного баланса |

In [ ]:
HF_p  = float(np.mean(HF_t))LF_p  = float(np.mean(LF_t))VLF_p = float(np.mean(VLF_t))TP_p  = HF_p + LF_p + VLF_pHFn   = HF_p  / TP_p * 100LFn   = LF_p  / TP_p * 100VLFn  = VLF_p / TP_p * 100LFHF  = LF_p / HF_pprint(f"=== Wavelet-derived powers ===")print(f"HF   = {HF_p:.2f}    ({HFn:.1f} %)")print(f"LF   = {LF_p:.2f}    ({LFn:.1f} %)")print(f"VLF  = {VLF_p:.2f}   ({VLFn:.1f} %)")print(f"TP   = {TP_p:.2f}")print(f"LF/HF = {LFHF:.2f}    [норма: 0.5–2]")

---## 6. Стандартные отклонения SDHF, SDLF, SDVLFСтандартные отклонения временных рядов `HF(t)`, `LF(t)`, `VLF(t)` показывают,**насколько устойчива** активность каждого регуляторного контура во времени:- Большие SDHF / SDLF / SDVLF → переходные процессы, нестационарность.- Малые → ритм стабильный.Эти показатели уникальны для вейвлет-анализа — FFT их не даёт.

In [ ]:
SDHF  = float(np.std(HF_t,  ddof=1))SDLF  = float(np.std(LF_t,  ddof=1))SDVLF = float(np.std(VLF_t, ddof=1))SDTP  = float(np.std(TP_t,  ddof=1))print(f"SDHF  = {SDHF:.2f}    (изменчивость HF во времени)")print(f"SDLF  = {SDLF:.2f}    (изменчивость LF во времени)")print(f"SDVLF = {SDVLF:.2f}   (изменчивость VLF во времени)")print(f"SDTP  = {SDTP:.2f}    (изменчивость суммарной мощности)")

---## 7. Сравнение FFT vs WaveletСделаем рядом FFT-оценку (как в Lab 1) и вейвлет-оценку нормированных мощностей.Различия покажут, как FFT-усреднение «съедает» временную динамику.

In [ ]:
# FFT-оценка (как в Lab 1)window  = np.hanning(N)rr_proc = (rr_uni - rr_uni.mean()) * windowX = np.fft.rfft(rr_proc)freqs_fft = np.fft.rfftfreq(N, d=dt)psd = np.abs(X)**2 / (fs * np.sum(window**2))psd[1:-1] *= 2def bp(flo, fhi):    m = (freqs_fft >= flo) & (freqs_fft <= fhi)    return float(np.trapezoid(psd[m], freqs_fft[m]))VLF_fft, LF_fft, HF_fft = bp(0.015, 0.040), bp(0.040, 0.150), bp(0.150, 0.400)TP_fft = VLF_fft + LF_fft + HF_fftnames = ['VLF', 'LF', 'HF']fft_pct = [VLF_fft/TP_fft*100, LF_fft/TP_fft*100, HF_fft/TP_fft*100]wt_pct  = [VLFn, LFn, HFn]fig, ax = plt.subplots(figsize=(8, 5))x = np.arange(len(names)); w = 0.36b1 = ax.bar(x - w/2, fft_pct, w, label='FFT', color='steelblue')b2 = ax.bar(x + w/2, wt_pct,  w, label='Wavelet (Morlet)', color='tomato')ax.set_xticks(x); ax.set_xticklabels(names)ax.set_ylabel('% от суммарной мощности')ax.set_title('Сравнение: FFT vs Вейвлет-разложение', fontsize=14, fontweight='bold')ax.legend(); ax.grid(axis='y', alpha=0.3)for b in list(b1) + list(b2):    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f'{b.get_height():.1f}%', ha='center', fontsize=9)plt.tight_layout()fig.savefig(FIGURES_DIR / 'fig3_fft_vs_wavelet.png')plt.show()print("\n=== Сравнение: FFT vs Wavelet ===")print(f"             FFT      Wavelet")print(f"VLF :  {fft_pct[0]:6.1f}%   {wt_pct[0]:6.1f}%")print(f"LF  :  {fft_pct[1]:6.1f}%   {wt_pct[1]:6.1f}%")print(f"HF  :  {fft_pct[2]:6.1f}%   {wt_pct[2]:6.1f}%")

---## 8. Сравнение разных базисных вейвлетов (опционально)Стандарт рекомендует для физиологии: Морле (`morl`), Гаусса (`gausN`), Добеши (`dbN`),симлеты (`symN`), койфлеты (`coifN`). Сравнение даёт интуицию, как выбор вейвлета влияетна разрешение по времени и частоте.

In [ ]:
wavelets_to_try = ['morl', 'gaus7', 'gaus8']  # Морле + Гаусс N=7,8results = []for w in wavelets_to_try:    _, _icwt, _F = get_wavelets(rr_uni, w, 0.015, 0.4, dt)    _power = np.abs(_icwt) ** 2 if False else np.abs(_icwt) ** 2  # placeholder    # quick mean-power check via cwt directly (icwt is band-weighted)    cwt_w, _, F_w = get_wavelets(rr_uni, w, 0.015, 0.4, dt)    P = np.abs(cwt_w) ** 2    HF_w  = band_timeseries(P, F_w, *bands['HF'])    LF_w  = band_timeseries(P, F_w, *bands['LF'])    VLF_w = band_timeseries(P, F_w, *bands['VLF'])    tp = float(np.mean(HF_w) + np.mean(LF_w) + np.mean(VLF_w))    results.append({        'wavelet': w,        'HFn':  np.mean(HF_w) / tp * 100,        'LFn':  np.mean(LF_w) / tp * 100,        'VLFn': np.mean(VLF_w) / tp * 100,        'LF/HF': np.mean(LF_w) / np.mean(HF_w),    })pd.DataFrame(results).round(2)

---## Сводка результатовВсе интегральные показатели в одной таблице:

In [ ]:
summary = pd.DataFrame({    'Показатель': ['HF', 'LF', 'VLF', 'TP', 'HFn (%)', 'LFn (%)', 'VLFn (%)',                   'LF/HF', 'SDHF', 'SDLF', 'SDVLF', 'SDTP'],    'Значение':   [f'{HF_p:.2f}', f'{LF_p:.2f}', f'{VLF_p:.2f}', f'{TP_p:.2f}',                   f'{HFn:.1f}', f'{LFn:.1f}', f'{VLFn:.1f}',                   f'{LFHF:.2f}', f'{SDHF:.2f}', f'{SDLF:.2f}', f'{SDVLF:.2f}', f'{SDTP:.2f}'],})summary